# Chicago Light Maps for the Bird-Collision Model
**Date:** 2026-09-30

## What this notebook does
We want to know if bird collisions are linked to artificial light. To test that, we first need **maps of light** that
we can match to collision locations.

This notebook makes four simple maps of Chicago from one satellite night photo, on a grid of 100 m x 100 m squares:

| Map | Question it answers |
|---|---|
| **Brightness** | How much light comes from this square? |
| **Lit share** | How much of this square is lit? |
| **Blueness** (B/G) | Is the light white/bluish? White LED lamps give high values. |
| **Redness** (R/G) | Is the light orange? Old sodium street lamps give high values. |

**Why color matters:** research suggests birds react more to blue-rich (white LED) light than to orange light. So we
look at *what kind* of light, not just *how much*.

## Step 0. Setup
The satellite is **SDGSAT-1**. It took this photo at about **9:30 pm on Sept 15, 2025**.
We use its **color image** (red, green, blue), where each pixel is 40 m wide. One file contains all of Chicago.

In [ ]:
import os, sys
# technical fix so map projections work in this Python environment (safe to ignore)
os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = os.path.join(sys.prefix, "share", "proj")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_origin
from rasterio.warp import reproject, transform_bounds, transform, Resampling

SCENE = "KX10_GIU_20250916_W89.00_N41.66_202500097282_L4A"
COLOR_FILE = f"{SCENE}/{SCENE}_B_RGB.tif"      # color image (camera B covers Chicago)
CALIB_FILE = f"{SCENE}/{SCENE}.calib.xml"      # numbers to convert pixel values into light units
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)

# west, south, east, north (longitude / latitude)
# east edge reaches -87.10 so the grid also covers the whole Lakefront Offshore Zone (it ends at -87.14)
CHICAGO = (-87.95, 41.62, -87.10, 42.05)

## Step 1. Read the image and convert it to light units
The file stores each pixel as a raw number from 0 to 4095. To compare red, green and blue fairly, we convert each raw
number into real light units (radiance) with a formula from the calibration file:

> light = raw number x gain + bias

A raw value of **0** means "no data" (outside the photo), so we mark it as empty (`NaN`).

In [ ]:
# read the gain and bias for each color from the calibration file
with open(CALIB_FILE, encoding="gb2312") as f:
    calib = ET.fromstring(f.read()).find(".//GIU")

def gain_bias(color):
    gain = float(calib.findtext(f".//RADIANCE_GAIN_BAND_{color}"))
    bias = float(calib.findtext(f".//RADIANCE_BIAS_BAND_{color}"))
    return gain, bias

# read only the Chicago part of the image
with rasterio.open(COLOR_FILE) as src:
    box = transform_bounds("EPSG:4326", src.crs, *CHICAGO)
    window = from_bounds(*box, transform=src.transform).round_offsets().round_lengths()
    raw = src.read(window=window).astype("float32")      # 3 layers: red, green, blue
    image_transform = src.window_transform(window)
    image_crs = src.crs

inside_photo = (raw > 0).all(axis=0)

light = {}
for i, color in enumerate(["RED", "GREEN", "BLUE"]):
    gain, bias = gain_bias(color)
    light[color] = np.where(inside_photo, raw[i] * gain + bias, np.nan)

print("Image size:", raw.shape[1], "x", raw.shape[2], "pixels (40 m each)")

## Step 2. Measure lit share
Green DN >= 20 defines lit pixels for the lit-share map only. Color calculations use all valid pixels, including dim, saturated, and extreme-color pixels. Only missing data are excluded.


In [ ]:
red, green, blue = light["RED"], light["GREEN"], light["BLUE"]

is_lit = inside_photo & (raw[1] >= 20)
print(f"Pixels inside the photo: {inside_photo.sum():,}")
print(f"Lit pixels (for lit share only): {is_lit.sum():,}")


## Step 3. Put everything on a 100 m grid
The collision model uses squares of **100 m x 100 m**, in the map system EPSG:26916. We use the same.

Each 100 m square covers parts of several 40 m pixels. For each square we take the **average** of the pixels inside it
(weighted by how much of each pixel falls in the square).

Then, for each square:
- **Brightness** = average total light (red + green + blue). We use a log scale because the city is thousands of times
  brighter than parks.
- **Lit share** = the fraction of valid covered area that is lit (green DN >= 20), from 0 to 1.
- **Blueness** = total blue / total green, using all valid pixels.
- **Redness** = total red / total green, using all valid pixels.

We divide *totals* instead of averaging each pixel's ratio, so bright lamps count more than faint, noisy pixels.
Color is calculated wherever valid data and a nonzero green denominator are available; no minimum lit share is required.

In [ ]:
GRID_CRS = "EPSG:26916"
CELL = 100   # meters

# grid covering Chicago, with edges on round 100 m numbers
west, south, east, north = transform_bounds("EPSG:4326", GRID_CRS, *CHICAGO)
west, south = np.floor(west / CELL) * CELL, np.floor(south / CELL) * CELL
east, north = np.ceil(east / CELL) * CELL, np.ceil(north / CELL) * CELL
grid_transform = from_origin(west, north, CELL, CELL)
grid_width, grid_height = int((east - west) / CELL), int((north - south) / CELL)

def to_100m(layer):
    # average a 40 m layer into the 100 m grid
    out = np.full((grid_height, grid_width), np.nan, dtype="float32")
    reproject(layer.astype("float32"), out,
              src_transform=image_transform, src_crs=image_crs, src_nodata=np.nan,
              dst_transform=grid_transform, dst_crs=GRID_CRS, dst_nodata=np.nan,
              resampling=Resampling.average)
    return out

total_light = to_100m(red + green + blue)
lit_share = to_100m(np.where(inside_photo, is_lit, np.nan))
red_100, green_100, blue_100 = to_100m(red), to_100m(green), to_100m(blue)

with np.errstate(divide="ignore", invalid="ignore"):
    maps = {
        "brightness": np.log10(total_light),
        "lit_share": lit_share,
        "blueness": np.where(green_100 != 0, blue_100 / green_100, np.nan),
        "redness": np.where(green_100 != 0, red_100 / green_100, np.nan),
    }

print(f"Grid: {grid_width} x {grid_height} = {grid_width * grid_height:,} squares of 100 m")
print(f"Squares with valid color: {np.isfinite(maps['blueness']).sum():,}")
pd.DataFrame({name: pd.Series(m.ravel()).describe() for name, m in maps.items()}).T.round(3)

## Step 4. Look at the maps

In [ ]:
extent_km = [west / 1000, east / 1000, south / 1000, north / 1000]

def centered_on_median(m):
    # color scale: white = typical Chicago value, stronger color = further from typical
    v = m[np.isfinite(m)]
    low, mid, high = np.percentile(v, [2, 50, 98])
    span = max(float(high - low), abs(float(mid)) * 0.01, 1e-6)
    return plt.matplotlib.colors.TwoSlopeNorm(
        vmin=min(low, mid - span * 0.01), vcenter=mid, vmax=max(high, mid + span * 0.01))

panels = [
    ("brightness", "Brightness\n(log scale)", "magma", None),
    ("lit_share", "Lit share\n(white = fully lit)", "Greys_r", None),
    ("blueness", "Blueness\n(blue = whiter, LED-like)", "RdBu",
     centered_on_median(maps["blueness"])),
    ("redness", "Redness\n(red = more orange, sodium-like)", "RdBu_r",
     centered_on_median(maps["redness"])),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 7), layout="constrained")

for ax, (name, title, cmap, norm) in zip(axes, panels):
    im = ax.imshow(
        maps[name],
        extent=extent_km,
        cmap=cmap,
        norm=norm,
        interpolation="nearest"
    )
    ax.set_facecolor("#1a1a19")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("km east")
    ax.set_ylabel("km north")
    plt.colorbar(im, ax=ax, shrink=0.7)

fig.savefig("figures/10_light_maps.png", dpi=150, bbox_inches="tight")
plt.show()

**How to read the maps:** brightness is log10 mean RGB radiance; lit share uses green DN >= 20. Color ratios include all valid pixels, including dark water and saturated or extreme values. Black indicates missing data. Display limits are chosen for visualization and do not remove values from the saved maps.


## Step 5. Threshold comparison (diagnostic only)
Compare color summaries for different lit definitions. These subsets do not filter the saved color maps, which use all valid pixels.


In [ ]:
on_photo = inside_photo
rows = []
for threshold in [10, 20, 50, 100]:
    for rule, pixels in [("green only (new)", on_photo & (raw[1] >= threshold)),
                         ("all three colors (old)", on_photo & (raw >= threshold).all(axis=0))]:
        rows.append({"threshold": threshold, "rule": rule,
                     "city blueness": blue[pixels].sum() / green[pixels].sum()})
check_a = pd.DataFrame(rows).pivot(index="threshold", columns="rule", values="city blueness").round(3)
check_a

This diagnostic compares selected lit subsets only. The saved color maps have no lit threshold or saturation/outlier filter.


## Step 6. Save the maps
Each map is saved as a GeoTIFF (an image that knows where it is on Earth), ready to join with collision locations.

In [ ]:
profile = {"driver": "GTiff", "dtype": "float32", "count": 1, "nodata": np.nan,
           "crs": GRID_CRS, "transform": grid_transform, "width": grid_width, "height": grid_height,
           "compress": "deflate"}
for name, m in maps.items():
    path = f"{OUT_DIR}/light_{name}_100m.tif"
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(m.astype("float32"), 1)
    print("saved", path)

## Step 7. Look up the light at any location
This is how collision records will get their light values: give longitudes and latitudes, and get back the value of
each map at those points. Here we try a few well-known places.

In [ ]:
def light_at(lon, lat):
    x, y = transform("EPSG:4326", GRID_CRS, list(lon), list(lat))
    rows, cols = rasterio.transform.rowcol(grid_transform, x, y)
    return pd.DataFrame({name: [m[r, c] for r, c in zip(rows, cols)] for name, m in maps.items()})

places = pd.DataFrame({
    "place": ["The Loop (downtown)", "McCormick Place", "Montrose Point bird sanctuary", "Lake Michigan, 3 km out"],
    "lon": [-87.6278, -87.6167, -87.6340, -87.5700],
    "lat": [41.8819, 41.8517, 41.9636, 41.9000],
})
pd.concat([places, light_at(places.lon, places.lat)], axis=1).round(3)

These example locations demonstrate how to retrieve map values. Color is now calculated for dark locations too, provided valid data exist. Rerun to see the updated values.


## What this means for the project
**Research question:** are bird collisions related to artificial light?

- We now have four light maps of Chicago at 100 m, on the same grid system as the collision model.
- Color ratios describe the relative RGB composition using all valid pixels. They are not verified lamp classifications.
- The maps also describe **current lighting in the Lakefront Protection Ordinance area**, which the final memo asks for,
  once we have the Public and Private Zone boundaries.

**Limits to keep in mind**
- This is **one night** (Sept 15, 2025, about 9:30 pm). Collisions were recorded from 2010 to 2024, and Chicago switched
  its streetlights from orange sodium to white LED between about 2017 and 2021.
- Blueness and redness are **hints** about lamp type, not proof. City streetlight records could confirm them.
- Collision records show **where dead birds were found**, not where risk is highest, because some places were searched
  much more than others.

**Next steps**
1. Get the collision file and Winnie's grid, then join light values to every collision (Step 7).
2. ~~Get the ordinance zone boundaries and compare light in the Public vs. Private Zone.~~ Done in
   `lakefront_zone_light.ipynb`.
3. Later, if useful: add the 500 m and 1000 m neighborhood versions that Winnie's model uses.